# 🎙️ Dagbani Whisper ASR — Live Inference & UI Studio

This notebook allows you to test your fine-tuned **Dagbani Whisper ASR** model (`ats-tech/dagbani-whisper-small`) with an interactive **Gradio Web Interface** on Google Colab or Kaggle.

### Features:
- 🎙️ **Live Microphone Recording:** Speak Dagbani directly into your microphone.
- 📁 **Audio File Upload:** Drag and drop WAV, MP3, M4A, FLAC, or OGG recordings.
- ✨ **Special Glyph Highlighting:** Automatically detects and reports Dagbani letters (`ɛ, ɔ, ŋ, ɣ, ʒ`).
- 🌐 **Public Shareable Link:** Generates a temporary public URL to test the app on mobile or share with teammates.
- 🧪 **Test-Split Benchmark Tool:** One-click transcription of ground-truth test clips directly from Hugging Face.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers gradio soundfile librosa huggingface_hub datasets

## 2. Hugging Face Authentication & Model Configuration

In [ ]:
import os
import torch
from huggingface_hub import login

# 1. Retrieve Hugging Face Token
HF_TOKEN = None

# Try Google Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

# Try Kaggle Secrets or Environment Variable
if not HF_TOKEN:
    HF_TOKEN = os.getenv('HF_TOKEN')

# If not found in secrets, paste your token below
if not HF_TOKEN:
    HF_TOKEN = ""  # <-- Paste your Hugging Face Token here if not using Colab Secrets

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Hugging Face Authentication successful.")
else:
    print("⚠️ No HF_TOKEN provided. If your model repo is private, please provide a token.")

# Set repository ID
MODEL_REPO_ID = "ats-tech/dagbani-whisper-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"🚀 Target Model: {MODEL_REPO_ID}")
print(f"⚡ Compute Device: {DEVICE.upper()} ({DTYPE})")

## 3. Load Model and Processor

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

print("📥 Downloading & Loading model weights from Hugging Face...")
processor = WhisperProcessor.from_pretrained(MODEL_REPO_ID, token=HF_TOKEN)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_REPO_ID,
    torch_dtype=DTYPE,
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
).to(DEVICE)

# Set model to inference mode and configure Whisper decoder for Dagbani
model.eval()
model.generation_config.forced_decoder_ids = None
model.generation_config.task = "transcribe"

print("✅ Model loaded successfully and ready for inference!")

## 4. Define the Inference Pipeline

In [ ]:
import librosa
import unicodedata

DAGBANI_GLYPHS = set("ɛɔŋɣʒƐƆŊƔƷ")

def transcribe_audio(audio_path):
    """Takes an audio file path, preprocesses to 16kHz mono, and generates Dagbani transcriptions."""
    if audio_path is None:
        return "⚠️ Please upload an audio file or record speech first.", "N/A"
    
    try:
        # Load and resample audio to 16 kHz mono (Whisper standard)
        waveform, sr = librosa.load(audio_path, sr=16_000, mono=True)
        duration = len(waveform) / 16_000
        
        # Extract Mel Spectrogram features
        inputs = processor(waveform, sampling_rate=16_000, return_tensors="pt", return_attention_mask=True)
        input_features = inputs.input_features.to(device=DEVICE, dtype=DTYPE)
        attention_mask = inputs.attention_mask.to(DEVICE)
        
        # Run inference
        with torch.inference_mode():
            predicted_ids = model.generate(
                input_features=input_features,
                attention_mask=attention_mask,
                max_length=225,
                num_beams=1,
            )
            
        # Decode tokens to Unicode NFC text
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        transcription = unicodedata.normalize("NFC", transcription).strip()
        
        # Analyze Dagbani glyphs in output
        detected_glyphs = sorted(set(ch for ch in transcription if ch in DAGBANI_GLYPHS))
        glyph_report = f"Found: {', '.join(detected_glyphs)}" if detected_glyphs else "None detected"
        
        metadata = f"⏱️ Duration: {duration:.2f}s | 🔤 Word Count: {len(transcription.split())} | ✨ Dagbani Glyphs: {glyph_report}"
        return transcription, metadata
        
    except Exception as exc:
        return f"❌ Error during inference: {str(exc)}", "Error"

## 5. Launch the Interactive Gradio Web Studio

In [ ]:
import gradio as gr

custom_css = """
#title { text-align: center; color: #1E3A8A; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
#output_box { font-size: 1.15rem; font-weight: 500; line-height: 1.6; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Dagbani Whisper ASR — Speech Studio", elem_id="title")
    gr.Markdown("Speak Dagbani into your microphone or upload an audio file to test real-time speech transcription.")
    
    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=["upload", "microphone"],
                type="filepath",
                label="🎙️ Record Audio or 📁 Upload File",
            )
            transcribe_btn = gr.Button("✨ Transcribe Speech", variant="primary", size="lg")
            
        with gr.Column(scale=1):
            text_output = gr.Textbox(
                label="📝 Dagbani Transcription",
                placeholder="Your transcription will appear here...",
                lines=4,
                elem_id="output_box",
            )
            metadata_output = gr.Markdown(label="📊 Clip Analytics")
            
    transcribe_btn.click(
        fn=transcribe_audio,
        inputs=[audio_input],
        outputs=[text_output, metadata_output],
    )

# share=True provides a public link accessible from any browser/phone
demo.launch(share=True, debug=True)

## 6. (Optional) Quick-Test with Real Test Clips from Hugging Face

Run this cell to stream a sample directly from the official **WAXAL Dagbani test split** and compare model output against ground-truth reference.

In [ ]:
from datasets import load_dataset
import soundfile as sf
import tempfile

print("📥 Fetching a test sample from WaxalNLP...")
test_stream = load_dataset("google/WaxalNLP", "dag_asr", split="test", streaming=True, token=HF_TOKEN)
sample = next(iter(test_stream))

audio_array = sample["audio"]["array"]
sample_rate = sample["audio"]["sampling_rate"]
ground_truth = sample.get("transcription", "N/A")

# Save temporary WAV
with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
    sf.write(tmp.name, audio_array, sample_rate)
    temp_path = tmp.name

# Run transcription
prediction, stats = transcribe_audio(temp_path)

print("=" * 60)
print(f"🎯 Reference (Ground Truth) : {ground_truth}")
print(f"🤖 Model Hypothesis         : {prediction}")
print(f"📊 Metadata                 : {stats}")
print("=" * 60)